# 02 Exploratory analysis

Everything here runs against `data/processed/rouanet.duckdb`, built by
`make processed`. The queries are the files in `sql/`, so the notebook and the
repository cannot disagree about what was computed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, duckdb
pd.set_option('display.width', 160)

In [ ]:
con = duckdb.connect('../data/processed/rouanet.duckdb', read_only=True)
con.execute('SELECT count(*) AS projetos, count(*) FILTER (WHERE in_model_sample) AS analysable FROM projetos').df()

## The headline

In [ ]:
s = con.execute('SELECT * FROM analysable').df()
r = s['share_raised']
pd.DataFrame({'threshold': ['raised nothing', '>0%', '>=20%', '>=50%', '>=100%'],
              'share': [(r<=0).mean(), (r>0).mean(), (r>=.2).mean(),
                        (r>=.5).mean(), (r>=1).mean()]}).round(3)

## Rate by category, with a confidence interval and a sample size

Wilson intervals rather than the normal approximation, because many cells are small.

In [ ]:
df = con.execute(open('../sql/q1_funding_rate_by_category.sql').read()).df()
df[df.dimensao == 'faixa_valor'].drop(columns='dimensao')

In [ ]:
df[df.dimensao == 'area'].drop(columns='dimensao')

In [ ]:
uf = df[df.dimensao == 'UF'].drop(columns='dimensao')
pd.concat([uf.head(5), uf.tail(5)])

## Over time, with the open cohorts flagged

The apparent fall after 2022 is censoring, not a trend: the projects that close
first are the ones that closed empty.

In [ ]:
con.execute(open('../sql/q1b_funding_rate_over_time.sql').read()).df()

## Concentration

In [ ]:
con.execute(open('../sql/q2_concentration_proponents.sql').read()).df()

In [ ]:
con.execute(open('../sql/q2b_concentration_sponsors.sql').read()).df()

In [ ]:
con.close()